# 03 · Validação estatística

**Objetivo:** verificar se os achados da EDA se sustentam estatisticamente: qual a incerteza do NPS geral, se a região muda o NPS e se o salto da 1ª para a 2ª reclamação é real.

**Entrada:** `data/nps_tratado.csv` · **Saída:** intervalo de confiança do NPS e testes de hipótese que sustentam as conclusões e os slides.

In [1]:
import numpy as np, pandas as pd
from scipy import stats
df = pd.read_csv("../data/nps_tratado.csv")
def nps(notas):
    return ((notas >= 9).mean() - (notas < 7).mean()) * 100

In [2]:
rng = np.random.default_rng(42)
notas = df["nps_score"].to_numpy()
boot = [nps(rng.choice(notas, size=len(notas))) for _ in range(5000)]
print(nps(notas), np.percentile(boot, [2.5, 97.5]))

-79.11338448422848 [-81.20204604 -77.06734868]


In [3]:
tab = pd.crosstab(df["customer_region"], df["nps_class"])
print("qui-quadrado p =", stats.chi2_contingency(tab)[1])
grupos = [g["nps_score"] for _, g in df.groupby("customer_region")]
print("Kruskal p =", stats.kruskal(*grupos)[1])

qui-quadrado p = 0.8778336172295353
Kruskal p = 0.3622841448288518


In [4]:
a = df.loc[df["complaints_count"] == 1, "nps_score"]
b = df.loc[df["complaints_count"] == 2, "nps_score"]
print("Mann-Whitney p =", stats.mannwhitneyu(a, b)[1])

Mann-Whitney p = 1.4351798071900387e-13


## Conclusões da validação

**Métodos:** bootstrap para o intervalo de confiança (aula 01) e testes de hipótese (aula 04): qui-quadrado e Kruskal-Wallis para região, Mann-Whitney para 1 vs 2 reclamações. Nível de significância de 5%.

- **NPS geral:** −79,1, com intervalo de confiança de 95% de aproximadamente −81,2 a −77,1. Mesmo no melhor cenario o NPS continua muito negativo.
- **Região:** não há evidência de diferença entre regiões (qui-quadrado p = 0,88; Kruskal-Wallis p =0,36). As diferenças vistas na EDA estão dentro do ruído, então região não deve guiar ações.
- **1 vs 2 reclamações:** a diferença de nota é estatisticamente sigificativa (Mann-Whitney p ≈ 1,4×10⁻¹³), e também grande na prática, vistoque os detratores saltam de 30% para 65%. Com base grande, p pequeno por si só não basta (aula 05), mas aqui o tamanho do efeito é relevante.
- **Limite:** os testes mostram que a diferença existe, não que as reclamações a causam. Os dados são observacionais; para validar a ação (contato na primeira reclamação) é preciso um teste A/B (aula 05).